
# <font color="green">Normal distribution</font>

## Background : Function calls

* If a function calls another function, its assembly becomes more complex, because:
  * calling a function with `bl` overwrites `x30` (the link register), so `x30` must be preserved on the stack;
  * that means the stack (`sp`) must be extended and the frame pointer (`x29`) set, so `x29` must be preserved too.
* In summary, a function that makes a call typically does something like
```
        stp     x29, x30, [sp, -16]!
```
to extend the stack and preserve `x29` and `x30` before the call, and restores them before returning.
* Observe this with `sigmoid.c`:
```
#include <math.h>
double sigmoid(double x) {
  return 1.0 / (1.0 + exp(-x));
}
```
compiled with `gcc -O3 -S sigmoid.c; cat sigmoid.s`.
* For details, study how a function call works in the [How Programming Languages Work (Basics)](https://taura.github.io/programming-languages/slides/05-implementation-basics.pdf) slide deck.

## A general framework for hand-compilation

* The problems below are too complex to tackle without a general framework. The main gaps between high-level languages and assembly are:
  * assembly has no structured compound statements, only branch instructions (≈ goto);
  * assembly does not allow nested expressions;
  * assembly has no new variables, only a fixed number of fixed-name variables (registers).
* Filling all three gaps at once is overwhelming. Instead, convert the program one step at a time:
  * convert loops and `if` statements into `goto`s;
  * break nested expressions into a series of simple assignments (`a * x + b * y` → `s = a * x; t = b * y; u = s + t`);
  * assign registers to variables.
* Also, when you call a function, save values you need after the call onto the stack.

## Problem

* Write a function `normal` that takes a floating-point (`double`) number $x$ and computes
$$ \mbox{normal}(x) \equiv \frac{1}{\sqrt{2\pi}}\exp(-x^2/2) $$
* To obtain $\pi$, use $\pi/4 = \mbox{atan2}(1.0, 1.0)$, i.e.
$$ \mbox{normal}(x) = \frac{1}{\sqrt{8 \;\mbox{atan2}(1.0, 1.0)}} \exp(-x^2/2) $$
* Fill in the skeleton `normal.s` (after `// ------- write your answer here -------`). You will need to call `exp`, `sqrt`, and `atan2` from the math library.
* The checker `check_normal.c` verifies your result (it is linked with `-lm`). If you see `OK`s and no errors, you are done.



# 1. AI Tutor
## 1-1. Prepare
* Your personal AI tutor is provided for questions and feedback.
* Execute the following cell before you use it.

In [ ]:
import heytutor

## 1-2. Examples
* A general question
```
%%hey
What does the `ldr` instruction do in ARM64?
```

* A hint on this specific problem
```
%%hey problem_file=normal.md
Give me a hint on this problem.

{problem}
```

* Builtin variables usable in `%%hey` cells
  * `{file:FILENAME}` is the content of FILE
  * `{bash[-1]}` is the output of the last `%%bash_` cell, `{bash[-2]}` the second last, etc.
  * `{problem}` is the content of the file you specify by `%%hey problem_file=foo.md`
  * `{answer}` is the content of the file you specify by `%%hey answer_file=foo.s`


# 2. Your Answer (assembly)
* Running the cell below writes the skeleton assembly file `normal.s`.
* Fill in your instructions after the line `// ------- write your answer here -------`, then run the cell again to save it.

In [ ]:
%%writefile_ normal.s
	.arch armv8-a
	.file	"normal.c"
	.text
	.align	2
	.p2align 4,,11
	.global	normal
	.type	normal, %function
normal:
.LFB0:
	.cfi_startproc
	// ------- write your answer here -------
	.cfi_endproc
.LFE0:
	.size	normal, .-normal
	.section	.rodata.cst8,"aM",@progbits,8
	.align	3
.LC0:
	.word	536225541
	.word	1074007443
	.ident	"GCC: (Ubuntu 13.3.0-6ubuntu2~24.04) 13.3.0"
	.section	.note.GNU-stack,"",@progbits


# 3. Checker
* The following C program calls your `normal` function and checks the result against a reference computed in C.

In [ ]:
%%writefile_ check_normal.c
#include <assert.h>
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
double normal(double x);

double normal_c(double x) {
  return exp(- x * x * 0.5) / sqrt(8.0 * atan2(1.0, 1.0));
}

int main(int argc, char ** argv) {
  assert(argc == 2);
  double x = atof(argv[1]);
  double y = normal(x);
  double yc = normal_c(x);
  if (fabs(y - yc) < 1.0e-6) {
    printf("OK %f %f\n", y, yc);
    return 0;
  } else {
    printf("NG %f %f\n", y, yc);
    return 1;
  }
}


# 4. Compile
* Compile your assembly together with the checker.
* If you get an error, fix `normal.s` above and recompile.

In [ ]:
%%bash_
gcc -o check_normal -O3 check_normal.c normal.s -lm


# 5. Run
* The commands to run the checker are stored in `run.sh`.
* If you see `OK`s and no errors, you are done.

In [ ]:
%%writefile_ run.sh
./check_normal 0.0
./check_normal 1.0
./check_normal 2.0

In [ ]:
%%bash_
bash run.sh


# 6. If things do not go well
* If your program compiles but does not produce the correct answer, run it within a debugger (gdb).
* Compile with `-O0 -g` first:
```
gcc -o check_normal -O0 -g check_normal.c normal.s -lm
```
* Then, in a terminal (SSH or Jupyter terminal):
```
gdb check_normal
(gdb) break normal
(gdb) run ...        # give the same arguments as in run.sh
```
* Step through one instruction at a time with `step`, and inspect registers with `print $x0` or `info registers`.

# 7. Ask Questions or Get Feedback
* You are encouraged to ask for feedback once you think you are done, to know if there is a better answer.

In [ ]:
%%hey problem_file=normal.md answer_file=normal.s

Problem:
{problem}

My Answer:
{answer}

Give me a feedback to my answer.